# Fine-tuning de LLM para análise de sentimento em PT-BR (GPU / CUDA)

**Modelo:** Qwen2.5-1.5B-Instruct + LoRA · **Dados:** B2W-Reviews01 · **GPU:** NVIDIA T4 (Google Colab)

Antes de começar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

> Para rodar dentro do VSCode, use a extensão **Google Colab** e selecione um servidor Colab com GPU como kernel (passo a passo em `docs/GPU_VSCODE.md`).

## 1. Verificar a GPU e o CUDA

In [ ]:
!nvidia-smi

## 2. Baixar o código e instalar dependências

In [ ]:
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"  # <-- troque pelo repositório do grupo
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 3. Informações da GPU (tire o print desta saída)

In [ ]:
!python src/gpu_info.py

## 4. Preparar os dados
Baixa o B2W-Reviews01 (132 mil avaliações), mapeia as notas em **negativo (1-2) / neutro (3) / positivo (4-5)** e cria splits balanceados:
3000 exemplos de treino, 300 de validação e 600 de teste.

In [ ]:
!python src/data.py

## 5. Baseline: modelo sem fine-tuning (zero-shot)
Mede o desempenho do Qwen2.5-1.5B-Instruct original, para comparação.

In [ ]:
!python src/evaluate.py --name baseline_zero_shot

## 6. Experimentos com diferentes parâmetros
Cada experimento muda **um** parâmetro em relação à referência (`r16_lr2e-4`: LoRA r=16, lr=2e-4, 1 época, todas as camadas lineares):

| Experimento | O que muda |
|---|---|
| `r4_lr2e-4`, `r64_lr2e-4` | rank do LoRA (capacidade do adaptador) |
| `r16_lr2e-5`, `r16_lr1e-3` | learning rate |
| `r16_lr2e-4_ep3` | número de épocas |
| `r16_lr2e-4_attn` | LoRA só nas camadas de atenção |

Tempo estimado na T4: ~1 hora no total. Para um teste rápido antes, rode `!python src/experiments.py --quick`.
Para rodar só alguns: `!python src/experiments.py --only r16_lr2e-4 r4_lr2e-4`.

In [ ]:
!python src/experiments.py --skip_baseline

## 7. Resultados

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/RESULTADOS.md").read()))
display(Image("results/comparacao.png"))
display(Image("results/curvas_loss.png"))

In [ ]:
import json, glob
best = max((json.load(open(p)) for p in glob.glob("results/*.json") if "f1_macro" in open(p).read()),
           key=lambda r: r["f1_macro"])
print("Melhor experimento:", best["name"], "| F1 macro:", best["f1_macro"])
display(Image(f"results/baseline_zero_shot_confusao.png"))
display(Image(f"results/{best['name']}_confusao.png"))

## 8. Análise qualitativa das respostas (erros do melhor modelo)

In [ ]:
import pandas as pd
pred = pd.read_csv(f"results/{best['name']}_predicoes.csv")
pd.set_option("display.max_colwidth", 200)
pred[pred.label != pred.pred][["text", "label", "pred", "confianca"]].head(15)

## 9. Salvar adaptadores e resultados no Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/llm_sentimento
!cp -r outputs results /content/drive/MyDrive/llm_sentimento/

## 10. Frontend
Abre a interface Gradio com o melhor adaptador. Clique no link público `*.gradio.live` que aparece na saída.

In [ ]:
!python app/app.py --adapter outputs/{best['name']} --share